In [11]:
import warnings
warnings.filterwarnings("ignore") # Mutes the yfinance pandas deprecation noise

import yfinance as yf
import pandas as pd
import sys
sys.path.append('../src')
from functions import merge_with_ff5

# Ensure these are real tickers. Start date must correspond with FF data in the functions.py.
tickers = ["AAPL", "MSFT" , "TSLA" , "TSM"] 
start_date = "2016-01-01"
end_date = "2026-10-31"

raw_data = yf.download(tickers, start=start_date, end=end_date)

# Dynamically select 'Adj Close' if it exists, otherwise fall back to 'Close'
price_col = 'Adj Close' if 'Adj Close' in raw_data else 'Close'

# Extract the prices safely
if len(tickers) > 1:
    adj_close = raw_data[price_col]
else:
    adj_close = raw_data[[price_col]].rename(columns={price_col: tickers[0]})
    
stock_rets = adj_close.resample('ME').last().pct_change().dropna()

# Process through SRC file function
regression_data = merge_with_ff5(stock_rets, tickers, start_date=start_date)
display(regression_data.tail())

[                       0%                       ]

[*********************100%***********************]  4 of 4 completed


,AAPL,MSFT,TSLA,TSM,Mkt-RF,SMB,HML,RMW,CMA,RF,AAPL_Excess,MSFT_Excess,TSLA_Excess,TSM_Excess
Date,,,,,,,,,,,,,,
2026-05-31,0.151087,0.106516,0.141918,0.056532,0.0491,-0.0265,-0.0231,-0.0816,-0.0146,0.0031,0.147987,0.103416,0.138818,0.053432
2026-06-30,-0.072742,-0.171509,-0.034856,0.143956,-0.0107,0.0436,0.0358,-0.0507,0.0345,0.0029,-0.075642,-0.174409,-0.037756,0.141056
2026-07-31,0.067563,0.245831,-0.260081,-0.153527,-0.0061,-0.0009,0.0211,0.1107,0.0227,0.0033,0.064263,0.242531,-0.263381,-0.156827
2026-08-31,0.026588,0.093659,0.182321,0.027384,0.0256,-0.0052,-0.0354,-0.0409,-0.0201,0.0029,0.023688,0.090759,0.179421,0.024484
2026-09-30,0.051034,0.011059,-0.035711,0.101353,-0.0107,-0.0460,-0.0155,-0.0322,0.0108,0.0031,0.047934,0.007959,-0.038811,0.098253


In [10]:
regression_data.to_csv("../data/raw/raw_regression_data.csv")